# 03 — Gold: current-version metrics

Read approved Silver Delta versions and rebuild detail, summaries, thresholds and review cases. Rebuilding aggregates removes stale contributions from corrected tickets, including changes of channel, employee or discount type. No files or full_/sample_ tables are read.

Run only after Silver COMPLETE, without concurrent writers. Dynamic checks compare outputs with current inputs; exact EDA totals remain optional regression checks. Failure leaves Gold RUNNING and publication unapproved.

In [ ]:
from pyspark.sql import functions as F
from pipeline_support import load_config, PipelineContext, assert_key, assert_same_keys
CFG = load_config()
CATALOG = CFG["catalog"]
SCHEMA = CFG["schemas"]["gold"]
K = ["CheckId", "LocationId", "BusinessDate"]
EXPECTED_TICKETS = CFG["original"]["tickets"]
EXPECTED_DAYS = CFG["original"]["days"]
TOLERANCE = float(CFG["tolerance"])
RUN_ORIGINAL_REGRESSION = CFG["run_original_regression"]
ctx = PipelineContext(spark, CFG, "gold")
ctx.begin()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
def table(name): return f"{CATALOG}.{SCHEMA}.{name}"
def save(df,name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table(name))
    return spark.table(table(name))
print("Destino:", CATALOG+"."+SCHEMA, "Run:", ctx.run_id)


## Definitions

Frequency = tickets with positive discounts / all tickets. Intensity = sum of discounts / sum of GrossPrice over the same valid population. Interpreted tip = −Gratuity over the final account excluding tip, for card-only payments. Accounts below 5 are flagged, not deleted; this cutoff is exploratory. Dashboard filters must sum numerators/denominators rather than average daily rates. Never average daily medians or percentiles.

In [ ]:
checks=ctx.read_input("workspace.applebees_silver.checks")
items=ctx.read_input("workspace.applebees_silver.items_sold")
payments=ctx.read_input("workspace.applebees_silver.payments")
discounts=ctx.read_input("workspace.applebees_silver.discounts")
taxes=ctx.read_input("workspace.applebees_silver.taxes")
voids=ctx.read_input("workspace.applebees_silver.voids")
d=discounts.groupBy(*K).agg(F.max(F.when(F.col("Amount")>0,1).otherwise(0)).alias("has_discount"),F.sum(F.when(F.col("discount_scope")=="ticket",F.col("Amount")).otherwise(0)).alias("ticket_discount_amount"),F.sum(F.when(F.col("discount_scope")=="item",F.col("Amount")).otherwise(0)).alias("item_discount_amount"))
p=payments.groupBy(*K).agg(F.count("*").alias("payment_count"),F.sum("Amount").alias("payment_amount_raw"),F.sum("Gratuity").alias("gratuity_raw"),F.max(F.when(F.col("Gratuity")<0,1).otherwise(0)).alias("has_negative_gratuity"))
i=items.groupBy(*K).agg(F.count("*").alias("item_line_count"),F.sum("GrossPrice").alias("grossprice_sum_unvalidated"),F.sum("SoldPrice").alias("soldprice_sum_unvalidated"))
v=voids.groupBy(*K).agg(F.count("*").alias("void_count"))
t=taxes.groupBy(*K).agg(F.sum("Amount").alias("tax_amount"))
facts=checks
for child in [d,p,i,v,t]:facts=facts.join(child,K,"left")
facts=facts.fillna(0,subset=["has_discount","payment_count","item_line_count","void_count","has_negative_gratuity"])
facts.createOrReplaceTempView("eda_input_facts")
for name,df in {"items":items,"payments":payments,"discounts":discounts,"taxes":taxes,"voids":voids}.items():df.createOrReplaceTempView("eda_input_"+name)
for alias,name in {"locations":"dim_locations","channels":"dim_revenue_centers","employees":"dim_employees"}.items():ctx.read_input("workspace.applebees_silver."+name).createOrReplaceTempView("eda_input_"+alias)
ctx.sql(f"CREATE OR REPLACE TEMP VIEW eda_parameters AS SELECT {TOLERANCE} AS tolerance, 100 AS min_employee_tickets,30 AS min_peer_tickets")


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TEMP VIEW eda_payment_summary AS
SELECT CheckId, LocationId, BusinessDate,
 COUNT(*) AS payment_records,
 CASE WHEN COUNT(DISTINCT PayCatName) = 1 THEN MAX(PayCatName) ELSE 'MULTIPLES/O_DESCONOCIDOS' END AS payment_method,
 CASE WHEN SUM(CASE WHEN IsCreditCard = true THEN 1 ELSE 0 END) = COUNT(*) THEN 'SOLO_TARJETA'
      WHEN SUM(CASE WHEN UPPER(PayCatName) = 'CASH' THEN 1 ELSE 0 END) = COUNT(*) THEN 'SOLO_EFECTIVO'
      ELSE 'OTROS_O_MIXTOS' END AS payment_family,
 SUM(CASE WHEN Gratuity > 0 THEN 1 ELSE 0 END) AS positive_tip_records,
 SUM(CASE WHEN Amount < 0 THEN 1 ELSE 0 END) AS negative_payment_records,
 COUNT(DISTINCT CASE WHEN EmployeeId > 0 THEN EmployeeId END) AS payment_employees,
 SUM(CASE WHEN EmployeeId IS NULL OR EmployeeId <= 0 THEN 1 ELSE 0 END) AS unknown_payment_employee_records,
 MAX(CASE WHEN EmployeeId > 0 THEN EmployeeId END) AS candidate_payment_employee
FROM eda_input_payments GROUP BY CheckId, LocationId, BusinessDate
""")
display(result)


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TEMP VIEW eda_void_summary AS
SELECT CheckId, LocationId, BusinessDate,
 SUM(Amount) AS void_amount_raw, SUM(Quantity) AS void_quantity_raw
FROM eda_input_voids GROUP BY CheckId, LocationId, BusinessDate
""")
display(result)


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TEMP VIEW eda_tickets AS
WITH base AS (
 SELECT f.*, l.Name AS location_name, l.TimeZoneName AS location_timezone, c.Name AS channel_name,
 p.payment_method, p.payment_family, p.positive_tip_records, p.negative_payment_records,
 p.payment_employees, p.unknown_payment_employee_records,
 CASE WHEN p.payment_employees = 1 AND p.unknown_payment_employee_records = 0
      THEN p.candidate_payment_employee END AS payment_employee_id,
 COALESCE(v.void_amount_raw, 0) AS void_amount_raw, COALESCE(v.void_quantity_raw, 0) AS void_quantity_raw,
 COALESCE(f.ticket_discount_amount, 0) + COALESCE(f.item_discount_amount, 0) AS discount_total_raw,
 f.soldprice_sum_unvalidated - COALESCE(f.ticket_discount_amount, 0) AS net_sales_candidate,
 f.soldprice_sum_unvalidated - COALESCE(f.ticket_discount_amount, 0)
   + COALESCE(f.tax_amount, 0) + COALESCE(f.ServiceCharge, 0) AS account_candidate,
 (f.payment_count > 0 AND f.item_line_count > 0 AND f.payment_amount_raw IS NOT NULL
  AND f.gratuity_raw IS NOT NULL AND f.soldprice_sum_unvalidated IS NOT NULL) AS assessable
 FROM eda_input_facts f
 LEFT JOIN eda_input_locations l ON f.LocationId = l.Id
 LEFT JOIN eda_input_channels c ON f.RevenueCenterId = c.Id
 LEFT JOIN eda_payment_summary p USING (CheckId, LocationId, BusinessDate)
 LEFT JOIN eda_void_summary v USING (CheckId, LocationId, BusinessDate)
), reconciled AS (
 SELECT *, payment_amount_raw + gratuity_raw - account_candidate AS reconciliation_error
 FROM base
), classified AS (
 SELECT *,
 CASE WHEN payment_count = 0 AND item_line_count = 0 THEN 'SIN_PRODUCTOS_NI_PAGOS'
      WHEN payment_count = 0 THEN 'SIN_PAGOS'
      WHEN item_line_count = 0 THEN 'SIN_PRODUCTOS'
      WHEN NOT assessable THEN 'IMPORTES_INCOMPLETOS'
      WHEN ABS(reconciliation_error) > p.tolerance THEN 'NO_CONCILIA'
      WHEN account_candidate <= 0 THEN 'CONCILIA_CUENTA_NO_POSITIVA'
      ELSE 'CONCILIA_CUENTA_POSITIVA' END AS account_status,
 COALESCE(assessable AND ABS(reconciliation_error) <= p.tolerance, false) AS reconciled,
 (grossprice_sum_unvalidated > 0 AND discount_total_raw >= 0
  AND discount_total_raw <= grossprice_sum_unvalidated + p.tolerance) AS discount_denominator_valid
 FROM reconciled CROSS JOIN eda_parameters p
)
SELECT *,
 CASE WHEN reconciled AND discount_denominator_valid
      THEN 100.0 * TRY_DIVIDE(discount_total_raw, grossprice_sum_unvalidated) END AS discount_pct,
 CASE WHEN reconciled AND account_candidate > 0 AND positive_tip_records = 0 AND gratuity_raw <= 0
      THEN -gratuity_raw END AS tip_amount_interpreted,
 CASE WHEN reconciled AND account_candidate > 0 AND positive_tip_records = 0 AND gratuity_raw <= 0
      THEN 100.0 * TRY_DIVIDE(-gratuity_raw, account_candidate) END AS tip_pct,
 COALESCE(reconciled AND account_candidate > 0 AND positive_tip_records = 0
          AND gratuity_raw <= 0 AND payment_family = 'SOLO_TARJETA', false) AS tip_comparable
FROM classified
""")
display(result)


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TABLE workspace.applebees_gold.ticket_metrics USING DELTA AS
SELECT t.*,q.mismatched_item_lines,q.unassessable_item_lines,q.missing_payments,q.missing_items,
 q.invalid_open_time,q.invalid_close_time,q.close_before_open,
 COALESCE(t.account_candidate>0 AND t.account_candidate<5,false) AS small_account,
 100.0*TRY_DIVIDE(t.tip_amount_interpreted,t.account_candidate+t.discount_total_raw) AS tip_pct_before_discount_proxy
FROM eda_tickets t JOIN workspace.applebees_silver.ticket_quality q USING(CheckId,LocationId,BusinessDate)
""")
print("Guardada: workspace.applebees_gold.ticket_metrics")


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TABLE workspace.applebees_gold.daily_location_channel USING DELTA AS
SELECT BusinessDate,LocationId,MAX(location_name) AS location_name,RevenueCenterId,MAX(channel_name) AS channel_name,
 COUNT(*) AS tickets,SUM(has_discount) AS discounted_tickets,
 SUM(CASE WHEN void_count>0 THEN 1 ELSE 0 END) AS tickets_with_voids,
 SUM(CASE WHEN reconciled THEN 1 ELSE 0 END) AS reconciled_tickets,
 SUM(CASE WHEN NOT assessable THEN 1 ELSE 0 END) AS incomplete_tickets,
 SUM(CASE WHEN assessable AND NOT reconciled THEN 1 ELSE 0 END) AS non_reconciled_tickets,
 SUM(CASE WHEN reconciled THEN net_sales_candidate ELSE 0 END) AS net_sales_reconciled,
 SUM(CASE WHEN reconciled THEN discount_total_raw ELSE 0 END) AS discounts_reconciled,
 SUM(CASE WHEN reconciled AND discount_denominator_valid THEN 1 ELSE 0 END) AS discount_ratio_tickets,
 SUM(CASE WHEN reconciled AND discount_denominator_valid THEN discount_total_raw ELSE 0 END) AS valid_discount_sum,
 SUM(CASE WHEN reconciled AND discount_denominator_valid THEN grossprice_sum_unvalidated ELSE 0 END) AS valid_gross_sum,
 SUM(CASE WHEN tip_comparable THEN 1 ELSE 0 END) AS tip_comparable_tickets,
 SUM(CASE WHEN tip_comparable THEN tip_amount_interpreted ELSE 0 END) AS comparable_tip_amount_sum,
 SUM(CASE WHEN tip_comparable THEN tip_pct ELSE 0 END) AS comparable_tip_pct_sum,
 SUM(CASE WHEN tip_comparable AND NOT small_account THEN 1 ELSE 0 END) AS tip_tickets_account_at_least_5,
 SUM(CASE WHEN tip_comparable AND NOT small_account THEN tip_pct ELSE 0 END) AS tip_pct_sum_account_at_least_5,
 SUM(mismatched_item_lines) AS mismatched_item_lines,
 SUM(CASE WHEN mismatched_item_lines>0 THEN 1 ELSE 0 END) AS tickets_with_item_mismatches
FROM workspace.applebees_gold.ticket_metrics GROUP BY BusinessDate,LocationId,RevenueCenterId
""")
print("Guardada: workspace.applebees_gold.daily_location_channel")


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TABLE workspace.applebees_gold.discount_type_daily USING DELTA AS
SELECT d.BusinessDate,d.LocationId,MAX(t.location_name) AS location_name,t.RevenueCenterId,
 MAX(t.channel_name) AS channel_name,d.discount_scope,d.DiscountId,d.DiscountName,
 COUNT(*) AS applications,SUM(CASE WHEN d.Amount>0 THEN 1 ELSE 0 END) AS positive_applications,
 COUNT(DISTINCT CASE WHEN d.Amount>0 THEN d.CheckId END) AS positive_discount_tickets,
 SUM(d.Amount) AS recorded_amount,SUM(CASE WHEN t.reconciled THEN d.Amount ELSE 0 END) AS amount_on_reconciled_tickets
FROM workspace.applebees_silver.discounts d JOIN workspace.applebees_gold.ticket_metrics t USING(CheckId,LocationId,BusinessDate)
GROUP BY d.BusinessDate,d.LocationId,t.RevenueCenterId,d.discount_scope,d.DiscountId,d.DiscountName
""")
print("Guardada: workspace.applebees_gold.discount_type_daily")


## Employees

A person can have multiple roles and a ticket can involve several people. Do not sum employee-associated sales as mutually exclusive sales. Ticket-level activity supports distinct counts by discount type or employee over filtered dates.

In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TEMP VIEW eda_employee_ticket AS
WITH operations AS (
 SELECT CheckId, LocationId, BusinessDate, EmployeeId, 1 AS item_role, 0 AS payment_role, 0 AS discount_role, 0 AS void_role FROM eda_input_items
 UNION ALL SELECT CheckId, LocationId, BusinessDate, EmployeeId, 0, 1, 0, 0 FROM eda_input_payments
 UNION ALL SELECT CheckId, LocationId, BusinessDate, EmployeeId, 0, 0, 1, 0 FROM eda_input_discounts
 UNION ALL SELECT CheckId, LocationId, BusinessDate, EmployeeId, 0, 0, 0, 1 FROM eda_input_voids
)
SELECT CheckId, LocationId, BusinessDate, EmployeeId,
 MAX(item_role) AS item_role, MAX(payment_role) AS payment_role, MAX(discount_role) AS discount_role, MAX(void_role) AS void_role
FROM operations WHERE EmployeeId > 0 GROUP BY CheckId, LocationId, BusinessDate, EmployeeId
""")
display(result)


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TEMP VIEW eda_employee_discounts AS
SELECT CheckId, LocationId, BusinessDate, EmployeeId,
 COUNT(*) AS discount_events, SUM(Amount) AS applied_discount_amount,
 MAX(CASE WHEN Amount > 0 THEN 1 ELSE 0 END) AS applied_positive_discount
FROM eda_input_discounts WHERE EmployeeId > 0 GROUP BY CheckId, LocationId, BusinessDate, EmployeeId
""")
display(result)


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TABLE workspace.applebees_gold.employee_ticket_activity USING DELTA AS
SELECT a.*,t.RevenueCenterId,t.channel_name,t.location_name,e.Deleted AS deleted_record,
 (e.Id IS NULL) AS unknown_employee,
 COALESCE(d.applied_positive_discount,0) AS applied_positive_discount,
 COALESCE(d.discount_events,0) AS own_discount_events,
 COALESCE(d.applied_discount_amount,0) AS own_recorded_discount_amount
FROM eda_employee_ticket a JOIN workspace.applebees_gold.ticket_metrics t USING(CheckId,LocationId,BusinessDate)
LEFT JOIN eda_employee_discounts d USING(CheckId,LocationId,BusinessDate,EmployeeId)
LEFT JOIN workspace.applebees_silver.dim_employees e ON a.EmployeeId=e.Id
""")
print("Guardada: workspace.applebees_gold.employee_ticket_activity")


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TABLE workspace.applebees_gold.employee_activity_daily USING DELTA AS
SELECT BusinessDate,LocationId,MAX(location_name) AS location_name,RevenueCenterId,MAX(channel_name) AS channel_name,EmployeeId,
 COUNT(*) AS associated_tickets,SUM(item_role) AS tickets_item_role,SUM(payment_role) AS tickets_payment_role,
 SUM(discount_role) AS tickets_discount_role,SUM(void_role) AS tickets_void_role,
 SUM(applied_positive_discount) AS tickets_with_own_positive_discount,
 SUM(CASE WHEN payment_role=1 THEN applied_positive_discount ELSE 0 END) AS payment_role_tickets_with_own_discount,
 SUM(own_discount_events) AS own_discount_events,SUM(own_recorded_discount_amount) AS own_recorded_discount_amount
FROM workspace.applebees_gold.employee_ticket_activity GROUP BY BusinessDate,LocationId,RevenueCenterId,EmployeeId
""")
print("Guardada: workspace.applebees_gold.employee_activity_daily")


## Review cases

Channel-specific P95 cutoffs use card payments, valid discount ratios and accounts of at least 5. The minimum of 30 tickets is exploratory. Small groups remain visible but do not produce P95 signals. Cutoffs use the entire period and do not automatically change with dashboard filters. Quality exceptions and small accounts with tips above 100% are also retained. No signal confirms fraud.

In [ ]:
from thresholds import materialize_thresholds
result = materialize_thresholds(ctx, lambda: ctx.sql("""
SELECT RevenueCenterId,MAX(channel_name) AS channel_name,COUNT(*) AS peer_tickets,
 PERCENTILE_APPROX(discount_pct,0.95) AS discount_p95,
 PERCENTILE_APPROX(tip_pct,0.95) AS tip_pct_p95,
 PERCENTILE_APPROX(tip_amount_interpreted,0.95) AS tip_amount_p95
FROM workspace.applebees_gold.ticket_metrics WHERE tip_comparable AND discount_pct IS NOT NULL AND account_candidate>=5
GROUP BY RevenueCenterId
"""), CODE_HASH, JOB_RUN_ID)


In [ ]:
result = ctx.sql("""
CREATE OR REPLACE TABLE workspace.applebees_gold.review_cases USING DELTA AS
WITH flags AS (
 SELECT t.*,p.peer_tickets,p.discount_p95,p.tip_pct_p95,p.tip_amount_p95,
 COALESCE(t.tip_comparable AND t.discount_pct IS NOT NULL AND t.account_candidate>=5 AND p.peer_tickets>=30
 AND t.discount_pct>p.discount_p95 AND t.tip_pct>p.tip_pct_p95,false) AS high_discount_and_tip,
 COALESCE(t.tip_comparable AND t.small_account AND t.tip_pct>100,false) AS small_account_high_tip,
 (t.account_status='NO_CONCILIA') AS non_reconciled_case,
 (t.mismatched_item_lines>0) AS item_mismatch_case
 FROM workspace.applebees_gold.ticket_metrics t LEFT JOIN workspace.applebees_gold.review_thresholds p ON t.RevenueCenterId=p.RevenueCenterId
)
SELECT *,COALESCE(high_discount_and_tip AND tip_amount_interpreted>tip_amount_p95,false) AS also_large_tip_amount
FROM flags WHERE high_discount_and_tip OR small_account_high_tip OR non_reconciled_case OR item_mismatch_case
""")
print("Guardada: workspace.applebees_gold.review_cases")


## Validate outputs against detail and EDA

Gold retains all tickets. Summary rows are reconciled against detail; historical references do not depend on old tables.

In [ ]:
metrics=spark.table(table("ticket_metrics"))
source=ctx.read_input("workspace.applebees_silver.checks")
assert_key(metrics,K,"Gold ticket_metrics")
assert_same_keys(source,metrics,K,"Silver vs Gold ticket_metrics")
SOURCE_TICKETS=source.count()
daily=spark.table(table("daily_location_channel"))
assert daily.agg(F.sum("tickets")).first()[0] == SOURCE_TICKETS,"Resumen de tickets no cuadra"
assert_key(spark.table(table("employee_ticket_activity")),K+["EmployeeId"],"employee_ticket_activity")
assert daily.agg(F.sum("discounted_tickets")).first()[0] == metrics.agg(F.sum("has_discount")).first()[0],"Descuentos no cuadran"
from decimal import Decimal
# Compare each numerator and denominator with current detail; do not average rates.
expected = {
    "tickets": F.count("*"),
    "discounted_tickets": F.sum("has_discount"),
    "tickets_with_voids": F.sum(F.when(F.col("void_count")>0,1).otherwise(0)),
    "reconciled_tickets": F.sum(F.when(F.col("reconciled"),1).otherwise(0)),
    "incomplete_tickets": F.sum(F.when(~F.col("assessable"),1).otherwise(0)),
    "non_reconciled_tickets": F.sum(F.when(F.col("assessable") & ~F.col("reconciled"),1).otherwise(0)),
    "net_sales_reconciled": F.sum(F.when(F.col("reconciled"),F.col("net_sales_candidate")).otherwise(0)),
    "discounts_reconciled": F.sum(F.when(F.col("reconciled"),F.col("discount_total_raw")).otherwise(0)),
    "discount_ratio_tickets": F.sum(F.when(F.col("reconciled") & F.col("discount_denominator_valid"),1).otherwise(0)),
    "valid_discount_sum": F.sum(F.when(F.col("reconciled") & F.col("discount_denominator_valid"),F.col("discount_total_raw")).otherwise(0)),
    "valid_gross_sum": F.sum(F.when(F.col("reconciled") & F.col("discount_denominator_valid"),F.col("grossprice_sum_unvalidated")).otherwise(0)),
    "tip_comparable_tickets": F.sum(F.when(F.col("tip_comparable"),1).otherwise(0)),
    "comparable_tip_amount_sum": F.sum(F.when(F.col("tip_comparable"),F.col("tip_amount_interpreted")).otherwise(0)),
    "comparable_tip_pct_sum": F.sum(F.when(F.col("tip_comparable"),F.col("tip_pct")).otherwise(0)),
    "tip_tickets_account_at_least_5": F.sum(F.when(F.col("tip_comparable") & ~F.col("small_account"),1).otherwise(0)),
    "tip_pct_sum_account_at_least_5": F.sum(F.when(F.col("tip_comparable") & ~F.col("small_account"),F.col("tip_pct")).otherwise(0)),
    "mismatched_item_lines": F.sum("mismatched_item_lines"),
    "tickets_with_item_mismatches": F.sum(F.when(F.col("mismatched_item_lines")>0,1).otherwise(0))
}
reference=metrics.agg(*[expr.alias(name) for name,expr in expected.items()]).first()
summary=daily.agg(*[F.sum(name).alias(name) for name in expected]).first()
for name in expected:
    a,b=reference[name],summary[name]
    assert (a is None and b is None) or (a is not None and b is not None and abs(Decimal(str(a))-Decimal(str(b))) <= Decimal(CFG["tolerance"])), f"Resumen distinto al detalle para {name}: {a} vs {b}"
applications=ctx.read_input("workspace.applebees_silver.discounts").count()
assert (spark.table(table("discount_type_daily")).agg(F.sum("applications")).first()[0] or 0) == applications, "Aplicaciones de descuento perdidas o duplicadas"
activity=spark.table(table("employee_ticket_activity")).count()
assert (spark.table(table("employee_activity_daily")).agg(F.sum("associated_tickets")).first()[0] or 0) == activity, "Actividad de empleados perdida o duplicada"
if RUN_ORIGINAL_REGRESSION:
    original=CFG["original"]
    for name in ["tickets","discounted_tickets","tickets_with_voids","non_reconciled_tickets","mismatched_item_lines"]:
        assert summary[name] == original[name], f"Regresión original distinta para {name}: {summary[name]} vs {original[name]}"
    for name in ["net_sales_reconciled","discounts_reconciled"]:
        assert abs(summary[name]-Decimal(original[name])) <= Decimal(CFG["tolerance"]), f"Regresión original distinta para {name}"
    print("Gold reproduce conteos e importes principales del dataset original.")
else:
    print("Regresión original conservada pero no aplicada a datos nuevos/corregidos.")
print("Gold validada contra las entradas vigentes. Completar relaciones antes de aprobar la capa.")


In [ ]:
result = ctx.sql("""
SELECT SUM(tickets) AS tickets,SUM(discounted_tickets) AS discounted_tickets,
 100.0*TRY_DIVIDE(SUM(discounted_tickets),SUM(tickets)) AS discount_ticket_pct,
 SUM(net_sales_reconciled) AS net_sales_reconciled,SUM(discounts_reconciled) AS discounts_reconciled,
 SUM(non_reconciled_tickets) AS non_reconciled_tickets,SUM(mismatched_item_lines) AS mismatched_item_lines
FROM workspace.applebees_gold.daily_location_channel
""")
display(result)


In [ ]:
result = ctx.sql("""
SELECT channel_name,COUNT(*) AS cases,SUM(CASE WHEN also_large_tip_amount THEN 1 ELSE 0 END) AS also_large_tip_cases FROM workspace.applebees_gold.review_cases GROUP BY channel_name
""")
display(result)


## Dashboard consumption

Use daily_location_channel for trends, discount_type_daily for types, employee_activity_daily for roles and review_cases for candidates. Recalculate filtered medians/percentiles from ticket_metrics. Type-specific affected-ticket counts overlap. Item quality flags distinguish validated amounts from exceptions. Currency is unconfirmed: label values as recorded amounts. Existing applebees_eda tables remain unchanged. The validated test Job chains the three notebooks and Audit; the existing dashboard has not been repointed.

## Keys and Catalog relationships

These cells run after tables are written and checked. They declare informational PK/FK constraints without RELY and retain pipeline controls. Primary keys require NOT NULL columns: nulls and uniqueness are checked before changing that property. Foreign keys require compatible types and matching parent records. Unmatched optional references, including EmployeeId/ManagerId=0, are reported and prevent declaring that whole-table FK. Employees are neither deleted nor reassigned. NULL represents absence and needs no parent.

Invalid keys or mandatory relationships stop execution. Unresolved optional relationships remain visible. Reruns recognize existing constraints without deleting them. Rebuilds require rerunning relationship blocks; rebuilding Silver also requires rebuilding Gold and its constraints. Bronze retains inputs without adding relational constraints.


In [ ]:
import re
from pyspark.sql import functions as F
relation_log = []
def ident(value):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), f"Identificador inesperado: {value}"
    return f"`{value}`"
def qualified(schema,name):
    return ".".join(ident(x) for x in ["workspace",schema,name])
def columns_sql(columns): return ", ".join(ident(x) for x in columns)
def existing_constraints(schema,name):
    return spark.sql(f"""SELECT constraint_name,constraint_type
    FROM workspace.information_schema.table_constraints
    WHERE table_catalog='workspace' AND table_schema='{schema}' AND table_name='{name}'""").collect()
def pk_columns(schema,name,constraint):
    return [r.column_name for r in spark.sql(f"""SELECT column_name FROM workspace.information_schema.key_column_usage
    WHERE constraint_catalog='workspace' AND constraint_schema='{schema}' AND constraint_name='{constraint}'
    AND table_schema='{schema}' AND table_name='{name}' ORDER BY ordinal_position""").collect()]
def add_pk(schema,name,columns,required=True):
    full=qualified(schema,name)
    df=spark.table(full)
    if df.filter(" OR ".join(f"{ident(c)} IS NULL" for c in columns)).limit(1).count():
        relation_log.append((schema,name,'PK','NO_DECLARADA: columnas nulas',','.join(columns)))
        if required: raise ValueError(f"Llave incompleta en {full}; investigar antes de declarar PK.")
        return False
    if df.groupBy(*columns).count().filter('count>1').limit(1).count():
        raise ValueError(f"Llave repetida en {full}; investigar antes de declarar PK.")
    pks=[r for r in existing_constraints(schema,name) if r.constraint_type=='PRIMARY KEY']
    if pks:
        assert pk_columns(schema,name,pks[0].constraint_name)==columns,f"PK existente diferente en {full}."
        relation_log.append((schema,name,'PK','YA_EXISTE',','.join(columns)))
        return True
    fields={f.name:f for f in df.schema.fields}
    for column in columns:
        if fields[column].nullable:
            spark.sql(f"ALTER TABLE {full} ALTER COLUMN {ident(column)} SET NOT NULL")
    constraint=f"pk_{schema}_{name}"
    spark.sql(f"ALTER TABLE {full} ADD CONSTRAINT {ident(constraint)} PRIMARY KEY ({columns_sql(columns)}) NOT ENFORCED")
    relation_log.append((schema,name,'PK','DECLARADA',','.join(columns)))
    return True

def add_fk(schema,name,columns,parent_schema,parent_name,parent_columns,label,required=False):
    full=qualified(schema,name);parent=qualified(parent_schema,parent_name)
    assert len(columns)==len(parent_columns)
    left=spark.table(full);right=spark.table(parent)
    parent_pks=[r for r in existing_constraints(parent_schema,parent_name) if r.constraint_type=='PRIMARY KEY']
    assert parent_pks and pk_columns(parent_schema,parent_name,parent_pks[0].constraint_name)==parent_columns, f"Falta PK compatible en {parent}. Ejecuta primero relaciones Silver."
    child_types={f.name:f.dataType for f in left.schema.fields};parent_types={f.name:f.dataType for f in right.schema.fields}
    assert all(child_types[c]==parent_types[p] for c,p in zip(columns,parent_columns)), f"Tipos distintos para FK de {full} hacia {parent}; revisar, no convertir automáticamente."
    # Require every fully non-null key combination to match a parent. NULL values represent missing references.
    observed=left.select(*columns).filter(" AND ".join(f"{ident(c)} IS NOT NULL" for c in columns))
    parents=right.select(*[F.col(p).alias(c) for c,p in zip(columns,parent_columns)])
    missing=observed.join(parents,columns,'left_anti').limit(1).collect()
    target=f"{parent_schema}.{parent_name}({','.join(parent_columns)})"
    if missing:
        relation_log.append((schema,name,'FK','NO_DECLARADA: referencia sin correspondencia',label+' -> '+target+'; ejemplo '+str(missing[0].asDict())))
        if required: raise ValueError(f"Referencia inválida de {full} hacia {parent}: {missing[0]}")
        return False
    constraint=f"fk_{schema}_{name}_{label}"
    if any(r.constraint_name==constraint and r.constraint_type=='FOREIGN KEY' for r in existing_constraints(schema,name)):
        relation_log.append((schema,name,'FK','YA_EXISTE',label+' -> '+target))
        return True
    spark.sql(f"ALTER TABLE {full} ADD CONSTRAINT {ident(constraint)} FOREIGN KEY ({columns_sql(columns)}) REFERENCES {parent} ({columns_sql(parent_columns)}) NOT ENFORCED")
    relation_log.append((schema,name,'FK','DECLARADA',label+' -> '+target))
    return True

def show_relations():
    display(spark.createDataFrame(relation_log,'schema STRING, table_name STRING, kind STRING, status STRING, detail STRING'))
    display(spark.sql("""SELECT table_schema,table_name,constraint_name,constraint_type
      FROM workspace.information_schema.table_constraints
      WHERE table_schema IN ('applebees_silver','applebees_gold') AND constraint_type IN ('PRIMARY KEY','FOREIGN KEY')
      ORDER BY table_schema,table_name,constraint_type"""))


In [ ]:
G='applebees_gold';S='applebees_silver';K=['CheckId','LocationId','BusinessDate']
add_pk(G,'ticket_metrics',K)
add_pk(G,'daily_location_channel',['BusinessDate','LocationId','RevenueCenterId'])
add_pk(G,'employee_ticket_activity',K+['EmployeeId'])
add_pk(G,'employee_activity_daily',['BusinessDate','LocationId','RevenueCenterId','EmployeeId'])
add_pk(G,'review_thresholds',['RevenueCenterId'])
add_pk(G,'review_cases',K)
add_pk(G,'discount_type_daily',['BusinessDate','LocationId','RevenueCenterId','discount_scope','DiscountId','DiscountName'],required=False)
add_fk(G,'ticket_metrics',K,S,'checks',K,'ticket',required=True)
for name in ['employee_ticket_activity','review_cases']:
    add_fk(G,name,K,G,'ticket_metrics',K,'ticket',required=True)
for name in ['ticket_metrics','daily_location_channel','employee_ticket_activity','employee_activity_daily','discount_type_daily','review_cases']:
    add_fk(G,name,['LocationId'],S,'dim_locations',['Id'],'location')
for name in ['ticket_metrics','daily_location_channel','employee_ticket_activity','employee_activity_daily','discount_type_daily','review_thresholds','review_cases']:
    add_fk(G,name,['RevenueCenterId'],S,'dim_revenue_centers',['Id'],'channel')
for name in ['employee_ticket_activity','employee_activity_daily']:
    add_fk(G,name,['EmployeeId'],S,'dim_employees',['Id'],'employee')
for name in ['ticket_metrics','review_cases']:
    add_fk(G,name,['payment_employee_id'],S,'dim_employees',['Id'],'payment_employee')
show_relations()


In [ ]:
ctx.finish([table(name) for name in [
    "ticket_metrics","daily_location_channel","discount_type_daily",
    "employee_ticket_activity","employee_activity_daily","review_thresholds","review_cases"]])
